In [ ]:
from collections.abc import Callable

import equinox as eqx
import jax
from context_flux_no.nn import Fourier1D


## TODO: Implement FNO and FluxFNO, test on Burgers to debug and test performance
class FNO1D(eqx.Module, strict=True):
    lift_layer: eqx.nn.MLP
    fourier_layers = tuple[Fourier1D, ...]
    project_layer: eqx.nn.MLP
    activation: Callable
    data_size: int = eqx.field(static=True)
    lift_size: int = eqx.field(static=True)
    depth: int = eqx.field(static=True)
    frequency_modes: int = eqx.field(static=True)
    stack_grid: bool = eqx.field(static=True)

    def __init__(
        self,
        data_size: int,
        lift_size: int,
        depth: int,
        frequency_modes: int,
        activation: Callable = jax.nn.gelu,
        stack_grid: bool = True,
        dtype=None,
        *,
        key,
    ):
        keys = jax.random.split(depth + 2)
        self.fourier_layers = tuple(
            Fourier1D(lift_size, lift_size, frequency_modes, activation, key=k)
            for k in keys[1:-1]
        )

    def layers(self) -> tuple[eqx.Module, ...]:
        return (self.lift_layer, *self.fourier_layers, self.project_layer)
